# 01.1 · 单体 Agent 的局限

> 为什么需要多 Agent？从单体架构的瓶颈说起。


In [1]:
import nest_asyncio; nest_asyncio.apply()
import queue, threading, asyncio, time, uuid, random, copy
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# ── 模拟 LLM / 向量检索 ─────────────────────────────────────────────────────
CORPUS = [
    "RAG 通过检索外部文档来扩充 LLM 的上下文。",
    "向量数据库支持语义相似度搜索。",
    "Reranker 用交叉编码器对候选文档重新打分。",
    "幂等设计让系统在重试时不产生副作用。",
    "Backpressure 防止下游服务因流量过大而崩溃。",
    "可观测性包括 Metrics、Traces 和 Logs 三大支柱。",
    "Graceful degradation 让系统在部分故障时仍能返回有用结果。",
    "Hub-and-Spoke 模式由 Orchestrator 统一分派任务。",
    "Blackboard 模式通过共享工作区实现多 Agent 协作。",
    "消息队列解耦生产者和消费者，支持异步处理。",
]

def fake_retrieve(query: str, top_k: int = 5, latency: float = 0.3) -> list[dict]:
    """模拟向量检索，返回 top_k 相关文档"""
    time.sleep(latency)
    scored = [(doc, random.uniform(0.5, 1.0)) for doc in CORPUS]
    scored.sort(key=lambda x: -x[1])
    return [{"text": doc, "score": round(s, 3)} for doc, s in scored[:top_k]]

def fake_rerank(query: str, hits: list[dict], top_k: int = 3, latency: float = 0.2) -> list[dict]:
    """模拟 cross-encoder reranker"""
    time.sleep(latency)
    reranked = copy.deepcopy(hits)
    for h in reranked:
        h["score"] = round(h["score"] * random.uniform(0.9, 1.1), 3)
    reranked.sort(key=lambda x: -x["score"])
    return reranked[:top_k]

def fake_generate(question: str, context: list[dict], latency: float = 0.5) -> str:
    """模拟 LLM 生成"""
    time.sleep(latency)
    snippets = " ".join(h["text"][:30] for h in context[:2])
    return f"[答案] 关于「{question[:20]}」：{snippets}..."

def fake_verify(answer: str, context: list[dict], latency: float = 0.15) -> dict:
    """模拟答案校验"""
    time.sleep(latency)
    return {"grounded": random.random() > 0.2, "confidence": round(random.uniform(0.7, 1.0), 2)}

print("✅ 模拟工具加载完毕")

✅ 模拟工具加载完毕


---
## Part 1 · 单体 Agent 的局限

所有逻辑塞在一个函数里：检索 → 重排 → 生成 → 校验，**串行执行**。

```
用户 ──▶ [ monolithic_agent() ] ──▶ 回答
            ├─ retrieve  (0.3s)
            ├─ rerank    (0.2s)
            ├─ generate  (0.5s)
            └─ verify    (0.15s)
               ──────────────────
               总计 ≈ 1.15s（串行叠加）
```

In [2]:
def monolithic_agent(query: str) -> dict:
    """单体 Agent：所有步骤串行执行"""
    t0 = time.perf_counter()

    hits    = fake_retrieve(query, top_k=5)
    topk    = fake_rerank(query, hits, top_k=3)
    answer  = fake_generate(query, topk)
    verdict = fake_verify(answer, topk)

    elapsed = round(time.perf_counter() - t0, 3)
    return {"answer": answer, "verdict": verdict, "latency_s": elapsed}

# ── 运行 ─────────────────────────────────────────────────────────────────────
result = monolithic_agent("什么是 RAG？")
print(f"延迟：{result['latency_s']}s")
print(f"校验：{result['verdict']}")
print(f"答案：{result['answer'][:80]}...")

延迟：1.151s
校验：{'grounded': False, 'confidence': 0.99}
答案：[答案] 关于「什么是 RAG？」：可观测性包括 Metrics、Traces 和 Logs 三 RAG 通过检索外部文档来扩充 LLM 的上下文。......


> **观察**：延迟 ≈ 各步骤之和。任何一步慢，整体就慢。
> 接下来我们拆分为多 Agent，观察延迟和可维护性的变化。

## 📖 讲义

# Part 1
## 引言：为什么需要多 Agent？

---

# 单体 Agent 的局限

```
用户 ──▶ [ 一个 LLM + 若干工具 ] ──▶ 回答
```

**在以下场景会遇到瓶颈：**

- **规模瓶颈**：检索 × 生成 × 验证串行，延迟叠加
- **职责模糊**：一个 Agent 承担过多角色，难以优化
- **无法独立扩容**：检索并发高但生成成本高，无法分别调配
- **故障扩散**：任一环节失败导致整体不可用
- **迭代困难**：换一个 Reranker 需要改动整个 Agent

---

# 多 Agent 带来什么？

```
用户
 │
 ▼
Orchestrator
 ├──▶ Retriever  (高并发，独立扩容)
 ├──▶ Reranker   (可替换实现)
 ├──▶ Generator  (成本精细控制)
 └──▶ Verifier   (独立故障隔离)
```

- **职责分离** — 每个 Agent 专注一件事，易测试、易替换
- **独立扩容** — 各服务按需弹性，成本可控
- **故障隔离** — Reranker 挂了可降级，不影响其他路径
- **并行执行** — 多个 Agent 同时工作，总延迟更短

---

---
**导航**：[📋 课程索引](./01_multi_agent_arch_demo.ipynb) · [01.2 · Pipeline 模式 →](./01.2_pipeline_mode.ipynb)
